# Type 2 diabetes versus coronary disease — the PCSK9 axis, systematically

Same three steps as `02_immunity_infection_pleiotropy.ipynb`, on the second trade-off the referee
names:

> Another example is PCSK9. The alleles at PCSK9 that increase a diagnosis of hypercholesterolemia
> should increase the risk of T2D. Is this not the case?

1. **Genes pleiotropic across the two disease classes** — at least one type 2 diabetes credible set
   and at least one coronary credible set.
2. **Same lead variant on both sides** — the only allele-exact comparison.
3. **PCSK9 specifically**, then a few representatives.

## Classes

The ontology does not give a usable single root for either side: `MONDO_0005148` (type 2 diabetes)
has only 2 descendants, and myocardial infarction, angina and CABG are *not* descendants of
`EFO_0001645` (coronary artery disease). Both classes are therefore built from an explicit list of
roots plus their descendants, and both are run in a narrow (primary) and a broad (sensitivity)
version:

| | primary | broad adds |
|---|---|---|
| **type 2 diabetes** | closure of `MONDO_0005148` type 2 diabetes and `EFO_0004997` type 2 diabetes nephropathy | as exact terms: `EFO_0000400` diabetes mellitus (unspecified), `EFO_0009406` glucose metabolism disease, `HP_0011014` abnormal glucose homeostasis, `HP_0003076` glycosuria, and the diabetic complications (retinopathy, neuropathy, nephropathy, eye disease, ketoacidosis, polyneuropathy, maculopathy) |
| **coronary** | closure of `EFO_0001645` coronary artery disease, `EFO_0000612` myocardial infarction, `EFO_0008583` acute MI, `MONDO_0021661` coronary atherosclerosis, `EFO_1001375` myocardial ischaemia, `EFO_0003913` angina pectoris, `EFO_1000985` intermediate coronary syndrome, `EFO_0003776` CABG, `EFO_0003951` PTCA | closure of `EFO_0003914` atherosclerosis, plus the exact terms `EFO_0003777` heart disease and `EFO_0000319` cardiovascular disease |

The broad additions are deliberately taken as **exact terms rather than closures**: the closure of
`EFO_0000400` (diabetes mellitus) contains type 1 and monogenic diabetes, and the closure of
`EFO_0000319` (cardiovascular disease) contains atrial fibrillation, hypertension, migraine and
stroke. Neither is the axis being tested. Type 1 diabetes, type 1 diabetic nephropathy, gestational
diabetes and cystic-fibrosis-related diabetes are excluded from both versions.

## Alleles, not variants

Every direction names an **effect allele**: the alternative allele of `chrom_pos_ref_alt`, the
harmonisation target at ingestion. Credible sets sharing a `variantId` share an effect allele, which
is why step 2 restricts the comparison to shared lead variants. Signed effect is
`rescaledStatistics.directionOfEffect × rescaledStatistics.absEstimatedBeta`; concordance is the
paper's definition (Methods).

As in notebook 02, credible sets from trans-disease / MTAG studies that map to a disease in **both**
classes ("Type 2 diabetes mellitus or coronary artery disease (pleiotropy)") are dropped from the
direction comparison — the same beta would otherwise appear on both sides.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 300)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"

T2D_ROOTS = ["MONDO_0005148", "EFO_0004997"]
# Broad additions are taken as exact terms, not closures: the closure of "diabetes mellitus" pulls in
# type 1 and monogenic diabetes, and the closure of "cardiovascular disease" pulls in atrial
# fibrillation, hypertension, migraine and stroke — neither is the axis being tested here.
T2D_BROAD_EXACT = [
    "EFO_0000400",  # diabetes mellitus, unspecified
    "EFO_0009406",  # glucose metabolism disease
    "HP_0011014",  # abnormal glucose homeostasis
    "HP_0003076",  # glycosuria
    "EFO_0003770",  # diabetic retinopathy
    "EFO_1000783",  # diabetic neuropathy
    "EFO_0000401",  # diabetic nephropathy
    "EFO_0009486",  # diabetic eye disease
    "EFO_1000897",  # diabetic ketoacidosis
    "MONDO_0001583",  # diabetic polyneuropathy
    "EFO_0010133",  # diabetic maculopathy
]
NOT_TYPE_2 = {"MONDO_0005147", "EFO_0004996", "EFO_0004593", "EFO_0801077"}

CAD_ROOTS = [
    "EFO_0001645",  # coronary artery disease
    "EFO_0000612",  # myocardial infarction
    "EFO_0008583",  # acute myocardial infarction
    "MONDO_0021661",  # coronary atherosclerosis
    "EFO_1001375",  # myocardial ischaemia
    "EFO_0003913",  # angina pectoris
    "EFO_1000985",  # intermediate coronary syndrome
    "EFO_0003776",  # coronary artery bypass
    "EFO_0003951",  # percutaneous transluminal coronary angioplasty
]
CAD_ROOTS_BROAD = CAD_ROOTS + [
    "EFO_0003914",  # atherosclerosis (closure: carotid, aortic, ...)
]
CAD_BROAD_EXACT = [
    "EFO_0003777",  # heart disease, unspecified
    "EFO_0000319",  # cardiovascular disease, unspecified
]

PCSK9_SYMBOL = "PCSK9"

In [2]:
cs = (
    ds.dataset(INTERMEDIATE + "qualifying_credible_sets", format="parquet")
    .to_table(
        columns=[
            "studyId",
            "studyLocusId",
            "variantId",
            "variant",
            "diseaseIds",
            "originalBeta",
            "originalStandardError",
            "rescaledStatistics",
            "studyStatistics",
            "nCases",
            "nControls",
        ]
    )
    .to_pandas()
)

rescaled = pd.DataFrame(list(cs["rescaledStatistics"]))
cs["beta"] = rescaled["directionOfEffect"].to_numpy() * rescaled["absEstimatedBeta"].to_numpy()
cs["se"] = rescaled["estimatedSE"].to_numpy()
cs["trait"] = pd.DataFrame(list(cs["studyStatistics"]))["trait"].to_numpy()
cs["effectAllele"] = cs["variant"].apply(lambda v: v["alt"])
cs["otherAllele"] = cs["variant"].apply(lambda v: v["ref"])
cs = cs.drop(columns=["variant", "rescaledStatistics", "studyStatistics"])

disease = pd.read_parquet(RELEASE + "output/disease", columns=["id", "name", "descendants"])
DISEASE_NAME = dict(zip(disease["id"], disease["name"]))
DESCENDANTS = {i: set(x) if x is not None else set() for i, x in zip(disease["id"], disease["descendants"])}

genes = pd.read_parquet(INTERMEDIATE + "list_of_prioritised_genes_per_CS.parquet")[["studyLocusId", "geneId"]]
target = pd.read_parquet(RELEASE + "output/target", columns=["id", "approvedSymbol"])
SYMBOL = dict(zip(target["id"], target["approvedSymbol"]))
GENE_ID = {v: k for k, v in SYMBOL.items()}

print(f"qualifying credible sets: {len(cs):,}")

qualifying credible sets: 70,618


In [3]:
def closure(roots: list[str]) -> set[str]:
    """Every root plus all of its descendants in the 25.06 disease index."""
    out: set[str] = set()
    for root in roots:
        if root not in DESCENDANTS:
            raise KeyError(f"{root} is not in the 25.06 disease index")
        out |= {root} | DESCENDANTS[root]
    return out


def paper_concordance(frame: pd.DataFrame) -> float:
    """Largest proportion of same-direction effects, over credible sets reporting a beta."""
    reported = frame.drop_duplicates("studyLocusId")
    reported = reported.loc[reported["originalBeta"].notna(), "beta"].dropna()
    if len(reported) == 0:
        return 1.0
    positive = float((reported > 0).mean())
    return max(positive, 1.0 - positive)


def median_beta(betas: pd.Series) -> float:
    """Median of the reported betas, NaN when none are reported."""
    reported = betas.dropna()
    return float(reported.median()) if len(reported) else float("nan")


def sign_of(betas: pd.Series) -> str:
    """Majority sign of a set of effect-allele betas: '+', '-' or 'mixed'."""
    reported = betas.dropna()
    positive, negative = int((reported > 0).sum()), int((reported < 0).sum())
    if positive > negative:
        return "+"
    if negative > positive:
        return "-"
    return "mixed"


T2D_TERMS = closure(T2D_ROOTS) - NOT_TYPE_2
T2D_TERMS_BROAD = (closure(T2D_ROOTS) | set(T2D_BROAD_EXACT)) - NOT_TYPE_2
CAD_TERMS = closure(CAD_ROOTS)
CAD_TERMS_BROAD = closure(CAD_ROOTS_BROAD) | set(CAD_BROAD_EXACT)

per_disease = cs.explode("diseaseIds").rename(columns={"diseaseIds": "diseaseId"}).dropna(subset=["diseaseId"])
per_disease["diseaseName"] = per_disease["diseaseId"].map(DISEASE_NAME)


def classify(frame: pd.DataFrame, t2d_terms: set[str], cad_terms: set[str]) -> pd.DataFrame:
    """Label each credible set x disease row as 't2d' or 'coronary', and flag joint studies."""
    out = frame.copy()
    out["class"] = np.where(
        out["diseaseId"].isin(t2d_terms),
        "t2d",
        np.where(out["diseaseId"].isin(cad_terms), "coronary", None),
    )
    out = out[out["class"].notna()].copy()
    joint = out.groupby("studyLocusId")["class"].nunique()
    out["joint_study"] = out["studyLocusId"].isin(set(joint[joint > 1].index))
    return out


classified = classify(per_disease, T2D_TERMS, CAD_TERMS)
classified_broad = classify(per_disease, T2D_TERMS_BROAD, CAD_TERMS_BROAD)

print(f"primary classes — type 2 diabetes: {len(T2D_TERMS)} terms, coronary: {len(CAD_TERMS)} terms")
print(f"broad classes   — type 2 diabetes: {len(T2D_TERMS_BROAD)} terms, coronary: {len(CAD_TERMS_BROAD)} terms")
print(
    f"\ncredible set x disease rows (primary) — t2d: {(classified['class'] == 't2d').sum():,}, "
    f"coronary: {(classified['class'] == 'coronary').sum():,}"
)
print(
    f"credible set x disease rows (broad)   — t2d: {(classified_broad['class'] == 't2d').sum():,}, "
    f"coronary: {(classified_broad['class'] == 'coronary').sum():,}"
)
print(
    f"\ntrans-disease / MTAG credible sets touching both classes (primary): "
    f"{classified.loc[classified['joint_study'], 'studyLocusId'].nunique()}"
)
print(classified.loc[classified["joint_study"], ["studyId", "trait"]].drop_duplicates().to_string(index=False))

primary classes — type 2 diabetes: 4 terms, coronary: 28 terms
broad classes   — type 2 diabetes: 15 terms, coronary: 37 terms

credible set x disease rows (primary) — t2d: 4,857, coronary: 3,264
credible set x disease rows (broad)   — t2d: 6,221, coronary: 4,007

trans-disease / MTAG credible sets touching both classes (primary): 90
     studyId                                                                             trait
GCST90451698                  Type 2 diabetes mellitus or coronary artery disease (pleiotropy)
GCST90451699 Type 2 diabetes mellitus adjusted for BMI or coronary artery disease (pleiotropy)
GCST90274723                                                    Cardiometabolic multimorbidity
  GCST010551                              Coronary heart disease x type 2 diabetes interaction


### The terms that actually carry data

In [4]:
for label, frame in [("primary", classified), ("broad", classified_broad)]:
    profile = (
        frame.groupby(["class", "diseaseId", "diseaseName"])
        .agg(credible_sets=("studyLocusId", "nunique"), lead_variants=("variantId", "nunique"))
        .reset_index()
        .sort_values(["class", "credible_sets"], ascending=[True, False])
    )
    print(f"===== {label} classes")
    print(profile.to_string(index=False))
    print()

===== primary classes
   class     diseaseId                                    diseaseName  credible_sets  lead_variants
coronary   EFO_0001645                        coronary artery disease           1438            974
coronary   EFO_0000612                          myocardial infarction            589            423
coronary   EFO_0003913                                angina pectoris            453            324
coronary MONDO_0021661                       coronary atherosclerosis            415            330
coronary   EFO_1001375                            Myocardial Ischemia            120            120
coronary   EFO_0003776                         coronary artery bypass             94             94
coronary   EFO_0003951 percutaneous transluminal coronary angioplasty             57             57
coronary   EFO_1000985                 intermediate coronary syndrome             50             47
coronary   EFO_0010820         spontaneous coronary artery dissection         

# Step 1 — genes pleiotropic across the two classes

In [5]:
def gene_table_for(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per L2G-prioritised gene with credible sets in both classes."""
    joined = frame.merge(genes, on="studyLocusId", how="inner")
    joined["symbol"] = joined["geneId"].map(SYMBOL)
    table = (
        joined.groupby(["geneId", "symbol"])
        .apply(
            lambda g: pd.Series(
                {
                    "t2d_credible_sets": g.loc[g["class"] == "t2d", "studyLocusId"].nunique(),
                    "coronary_credible_sets": g.loc[g["class"] == "coronary", "studyLocusId"].nunique(),
                    "t2d_diseases": g.loc[g["class"] == "t2d", "diseaseId"].nunique(),
                    "coronary_diseases": g.loc[g["class"] == "coronary", "diseaseId"].nunique(),
                    "lead_variants": g["variantId"].nunique(),
                }
            ),
            include_groups=False,
        )
        .reset_index()
    )
    return table[(table["t2d_credible_sets"] > 0) & (table["coronary_credible_sets"] > 0)]


with_genes = classified.merge(genes, on="studyLocusId", how="inner")
with_genes["symbol"] = with_genes["geneId"].map(SYMBOL)
with_genes_broad = classified_broad.merge(genes, on="studyLocusId", how="inner")
with_genes_broad["symbol"] = with_genes_broad["geneId"].map(SYMBOL)

gene_table = gene_table_for(classified).sort_values(["t2d_credible_sets", "coronary_credible_sets"], ascending=False)
gene_table_broad = gene_table_for(classified_broad)

gene_table.to_csv(INTERMEDIATE + "t2d_cad_genes-r1.csv", index=False)

print(f"genes with credible sets in both classes — primary: {len(gene_table)}, broad: {len(gene_table_broad)}")
print()
print(gene_table.head(25).to_string(index=False))

genes with credible sets in both classes — primary: 160, broad: 205

         geneId  symbol  t2d_credible_sets  coronary_credible_sets  t2d_diseases  coronary_diseases  lead_variants
ENSG00000148737  TCF7L2                 89                       4             2                  2             24
ENSG00000147883  CDKN2B                 66                     109             2                  9             69
ENSG00000145996  CDKAL1                 64                       1             2                  1             29
ENSG00000073792 IGF2BP2                 54                       1             2                  1             30
ENSG00000140718     FTO                 41                       4             2                  2             24
ENSG00000152804    HHEX                 41                       2             2                  1             26
ENSG00000118971   CCND2                 41                       1             2                  1             16
ENSG0000015

# Step 2 — the same lead variant on both sides

In [6]:
def shared_variants(frame: pd.DataFrame) -> pd.DataFrame:
    """Gene x lead-variant pairs carrying both classes, excluding trans-disease credible sets."""
    comparable = frame[~frame["joint_study"]]
    table = (
        comparable.groupby(["geneId", "symbol", "variantId", "effectAllele", "otherAllele"])
        .apply(
            lambda g: pd.Series(
                {
                    "t2d_credible_sets": g.loc[g["class"] == "t2d", "studyLocusId"].nunique(),
                    "coronary_credible_sets": g.loc[g["class"] == "coronary", "studyLocusId"].nunique(),
                    "t2d_diseases": g.loc[g["class"] == "t2d", "diseaseId"].nunique(),
                    "coronary_diseases": g.loc[g["class"] == "coronary", "diseaseId"].nunique(),
                    "t2d_sign": sign_of(g.loc[g["class"] == "t2d", "beta"]),
                    "coronary_sign": sign_of(g.loc[g["class"] == "coronary", "beta"]),
                    "t2d_median_beta": median_beta(g.loc[g["class"] == "t2d", "beta"]),
                    "coronary_median_beta": median_beta(g.loc[g["class"] == "coronary", "beta"]),
                }
            ),
            include_groups=False,
        )
        .reset_index()
    )
    table = table[(table["t2d_credible_sets"] > 0) & (table["coronary_credible_sets"] > 0)].copy()
    table["verdict"] = np.where(
        (table["t2d_sign"] == "mixed") | (table["coronary_sign"] == "mixed"),
        "undetermined",
        np.where(table["t2d_sign"] == table["coronary_sign"], "concordant", "discordant"),
    )
    table["variant_concordance_paper_formula"] = (
        table["variantId"].map(cs.groupby("variantId").apply(paper_concordance, include_groups=False)).round(3)
    )
    return table.sort_values(["t2d_credible_sets", "coronary_credible_sets"], ascending=False)


shared = shared_variants(with_genes)
shared_broad = shared_variants(with_genes_broad)
shared.to_csv(INTERMEDIATE + "t2d_cad_shared_variants-r1.csv", index=False)
shared_broad.to_csv(INTERMEDIATE + "t2d_cad_shared_variants_broad-r1.csv", index=False)

for label, table in [("primary", shared), ("broad", shared_broad)]:
    determined = (table["verdict"] != "undetermined").sum()
    print(
        f"{label}: {len(table)} gene x lead-variant pairs, {table['geneId'].nunique()} genes, "
        f"{table['variantId'].nunique()} lead variants"
    )
    print(
        f"  {table['verdict'].value_counts().to_dict()}   "
        f"discordant share of determined: "
        f"{(table['verdict'] == 'discordant').sum() / max(determined, 1):.1%}"
    )

primary: 27 gene x lead-variant pairs, 23 genes, 27 lead variants
  {'concordant': 15, 'undetermined': 9, 'discordant': 3}   discordant share of determined: 16.7%
broad: 37 gene x lead-variant pairs, 29 genes, 37 lead variants
  {'concordant': 25, 'undetermined': 8, 'discordant': 4}   discordant share of determined: 13.8%


In [7]:
print("primary classes — every gene x lead-variant pair:")
print(
    shared[
        [
            "symbol",
            "variantId",
            "effectAllele",
            "t2d_credible_sets",
            "coronary_credible_sets",
            "t2d_sign",
            "coronary_sign",
            "t2d_median_beta",
            "coronary_median_beta",
            "verdict",
            "variant_concordance_paper_formula",
        ]
    ].to_string(index=False)
)

primary classes — every gene x lead-variant pair:
  symbol        variantId effectAllele  t2d_credible_sets  coronary_credible_sets t2d_sign coronary_sign  t2d_median_beta  coronary_median_beta      verdict  variant_concordance_paper_formula
  TCF7L2 10_112998590_C_T            T                 42                       2        +             +         0.274212              0.024855   concordant                              0.951
    APOE  19_44908684_T_C            C                  9                      11        -             +        -0.059317              0.081001   discordant                              0.659
     FTO  16_53767042_T_C            C                  9                       2        +             +         0.126584              0.039061   concordant                              0.945
   GRB14  2_164672366_C_T            T                  4                       2        -             -        -0.057528             -0.034736   concordant                          

In [8]:
print("broad classes — the discordant pairs (the pattern the referee predicts):")
print(
    shared_broad[shared_broad["verdict"] == "discordant"][
        [
            "symbol",
            "variantId",
            "effectAllele",
            "t2d_credible_sets",
            "coronary_credible_sets",
            "t2d_sign",
            "coronary_sign",
            "t2d_median_beta",
            "coronary_median_beta",
            "variant_concordance_paper_formula",
        ]
    ].to_string(index=False)
)

broad classes — the discordant pairs (the pattern the referee predicts):
symbol        variantId effectAllele  t2d_credible_sets  coronary_credible_sets t2d_sign coronary_sign  t2d_median_beta  coronary_median_beta  variant_concordance_paper_formula
  APOE  19_44908684_T_C            C                 16                      14        -             +        -0.059934              0.076785                              0.659
PNPLA3  22_43928850_C_T            T                  3                       2        +             -         0.060906             -0.032832                              0.697
  APOE  19_44919689_A_G            G                  1                       5        -             +        -0.059933              0.061749                              0.556
  CUX2 12_111269073_C_T            T                  1                       4        +             -         0.031964             -0.129644                              0.571


# Step 3a — PCSK9 itself

The referee's example, at gene level rather than at one variant: every credible set whose
L2G-prioritised gene is PCSK9.

In [9]:
pcsk9_loci = genes[genes["geneId"] == GENE_ID[PCSK9_SYMBOL]]["studyLocusId"]
pcsk9 = per_disease[per_disease["studyLocusId"].isin(set(pcsk9_loci))].copy()

pcsk9_profile = (
    pcsk9.groupby(["diseaseId", "diseaseName"])
    .agg(
        credible_sets=("studyLocusId", "nunique"),
        lead_variants=("variantId", "nunique"),
        n_increases=("beta", lambda s: int((s > 0).sum())),
        n_decreases=("beta", lambda s: int((s < 0).sum())),
        min_beta=("beta", "min"),
        max_beta=("beta", "max"),
    )
    .reset_index()
    .sort_values("credible_sets", ascending=False)
)
pcsk9_profile["class_primary"] = np.where(
    pcsk9_profile["diseaseId"].isin(T2D_TERMS),
    "t2d",
    np.where(pcsk9_profile["diseaseId"].isin(CAD_TERMS), "coronary", "-"),
)
pcsk9_profile["class_broad"] = np.where(
    pcsk9_profile["diseaseId"].isin(T2D_TERMS_BROAD),
    "t2d",
    np.where(pcsk9_profile["diseaseId"].isin(CAD_TERMS_BROAD), "coronary", "-"),
)
pcsk9_profile.to_csv(INTERMEDIATE + "t2d_cad_pcsk9_profile-r1.csv", index=False)

print(f"credible sets with PCSK9 as the L2G-prioritised gene: {pcsk9['studyLocusId'].nunique()}")
print(f"distinct lead variants: {pcsk9['variantId'].nunique()}")
print(
    f"diabetes / glycaemic credible sets — primary class: "
    f"{pcsk9['diseaseId'].isin(T2D_TERMS).sum()}, broad class: "
    f"{pcsk9['diseaseId'].isin(T2D_TERMS_BROAD).sum()}"
)
print(
    f"coronary credible sets — primary class: {pcsk9['diseaseId'].isin(CAD_TERMS).sum()}, "
    f"broad class: {pcsk9['diseaseId'].isin(CAD_TERMS_BROAD).sum()}"
)
print()
print(pcsk9_profile.to_string(index=False))

credible sets with PCSK9 as the L2G-prioritised gene: 109
distinct lead variants: 24
diabetes / glycaemic credible sets — primary class: 0, broad class: 0
coronary credible sets — primary class: 40, broad class: 48

      diseaseId                                    diseaseName  credible_sets  lead_variants  n_increases  n_decreases  min_beta  max_beta class_primary class_broad
    EFO_0001645                        coronary artery disease             16              6            1           14 -0.373599  0.023544      coronary    coronary
     HP_0003124                           Hypercholesterolemia             15              8            2           13 -1.054551  0.049608             -           -
    EFO_0000612                          myocardial infarction              9              5            1            7 -0.375070  0.068968      coronary    coronary
     HP_0003077                                 Hyperlipidemia              8              7            1            7 -0.98

### The one diabetes-mapped credible set at PCSK9 in the full release

The Open Targets Platform does show a diabetes credible set on `1_55039974_G_T`, so it is worth being
precise about what our corpus drops and why. The next cell searches the **whole 25.06 credible-set
release** (not the qualifying subset) in the PCSK9 window for any credible set whose study maps to a
diabetes or glycaemic term.

In [10]:
release_cs = (
    ds.dataset(RELEASE + "output/credible_set", format="parquet")
    .to_table(
        columns=[
            "studyLocusId",
            "studyId",
            "variantId",
            "beta",
            "standardError",
            "pValueMantissa",
            "pValueExponent",
            "finemappingMethod",
            "confidence",
        ],
        filter=(
            (pc.field("chromosome") == "1")
            & (pc.field("position") > 54_500_000)
            & (pc.field("position") < 55_600_000)
            & (pc.field("studyType") == "gwas")
        ),
    )
    .to_pandas()
)

release_studies = (
    ds.dataset(RELEASE + "output/study", format="parquet")
    .to_table(
        columns=["studyId", "traitFromSource", "traitFromSourceMappedIds", "nCases", "nControls", "nSamples"],
    )
    .to_pandas()
)

window = release_cs.merge(release_studies, on="studyId", how="inner")
window["diabetes_terms"] = window["traitFromSourceMappedIds"].apply(
    lambda ids: [t for t in (ids if ids is not None else []) if t in T2D_TERMS_BROAD]
)
diabetes_at_pcsk9 = window[window["diabetes_terms"].str.len() > 0].copy()
diabetes_at_pcsk9["diabetes_terms"] = diabetes_at_pcsk9["diabetes_terms"].apply(
    lambda ids: ", ".join(f"{t} ({DISEASE_NAME.get(t)})" for t in ids)
)
l2g_symbols = (
    genes.assign(symbol=genes["geneId"].map(SYMBOL))
    .groupby("studyLocusId")["symbol"]
    .apply(lambda s: ", ".join(sorted(set(s.dropna()))))
)
diabetes_at_pcsk9["l2g_gene"] = diabetes_at_pcsk9["studyLocusId"].map(l2g_symbols)
diabetes_at_pcsk9["in_qualifying_corpus"] = diabetes_at_pcsk9["studyLocusId"].isin(set(cs["studyLocusId"]))
diabetes_at_pcsk9.to_csv(INTERMEDIATE + "t2d_cad_pcsk9_release_diabetes-r1.csv", index=False)

print(f"GWAS credible sets in PCSK9 +/- 500 kb, whole 25.06 release: {len(window):,}")
print(f"of those mapped to a diabetes / glycaemic term: {len(diabetes_at_pcsk9)}")
print()
print(
    diabetes_at_pcsk9[
        [
            "variantId",
            "l2g_gene",
            "studyId",
            "traitFromSource",
            "diabetes_terms",
            "beta",
            "pValueMantissa",
            "pValueExponent",
            "nCases",
            "nControls",
            "nSamples",
            "finemappingMethod",
            "in_qualifying_corpus",
        ]
    ].to_string(index=False)
)

GWAS credible sets in PCSK9 +/- 500 kb, whole 25.06 release: 3,679
of those mapped to a diabetes / glycaemic term: 2

     variantId l2g_gene      studyId                                   traitFromSource                  diabetes_terms      beta  pValueMantissa  pValueExponent  nCases  nControls  nSamples finemappingMethod  in_qualifying_corpus
1_55039974_G_T    PCSK9 GCST90309362 Diabetes (confirmatory factor analysis Factor 28) EFO_0000400 (diabetes mellitus) -0.009231        8.080789             -11     0.0        0.0  360514.0         SuSiE-inf                 False
1_55407443_T_C    USP24 GCST90296598                                          Diabetes EFO_0000400 (diabetes mellitus) 23.737800        6.659000             -10   291.0     5443.0    5734.0              PICS                 False


Exactly one of them is on `rs11591147` itself: **GCST90309362**, "Diabetes (confirmatory factor
analysis Factor 28)" (Carey et al. 2024), mapped to `EFO_0000400` diabetes mellitus, β = **−0.0092**
for the **T** allele, p = 8.1 × 10⁻¹¹, L2G-assigned to PCSK9. It is excluded from our corpus by the
study filter, not by anything variant-specific: the study reports **0 cases and 0 controls** on
360,514 samples, so it fails the `binaryLessCases` requirement — it is a continuous latent
factor score from a confirmatory factor analysis, not a case/control diabetes diagnosis. All 34
Carey factor studies in the release are dropped the same way.

Two things follow. First, the coverage statement stands as a statement about the *qualifying*
corpus, and the exclusion is a deliberate study-level rule rather than a gap at this locus. Second,
even that credible set does not support the predicted trade-off: the T allele, which lowers
hypercholesterolaemia and coronary risk, has a **negative** effect on the diabetes factor as well.
The other window hit, `1_55407443_T_C` (GCST90296598, 291 cases), is 370 kb away, L2G-assigned to
USP24 rather than PCSK9, and is also outside the qualifying corpus.

**PCSK9 cannot answer the question.** Not one of its qualifying credible sets — across all 24 lead
variants — is mapped to type 2 diabetes or to any glycaemic term, even under the broad class; the
gene's qualifying associations are lipid, coronary and vascular only. The single diabetes-mapped
credible set that exists in the wider release is the Carey factor-analysis study above, excluded at
study level, and it points the same protective way. The trade-off the referee predicts is therefore
untestable at PCSK9 here: missing data on the diabetes side, not a concordant result.

# Step 3b — representatives

Chosen from the shared-lead-variant tables to cover both verdicts, with the discordant ones being
exactly the pattern the referee describes: an allele that lowers diabetes risk and raises coronary
risk, or the reverse.

In [11]:
REPRESENTATIVES = [
    (
        "APOE (rs429358 region)",
        "19_44908684_T_C",
        "lowers type 2 diabetes risk, raises coronary risk — the referee's pattern",
    ),
    ("APOE (second lead variant)", "19_44919689_A_G", "same pattern on an independent lead variant of the locus"),
    ("PNPLA3", "22_43928850_C_T", "raises diabetes risk, lowers coronary risk — the reverse trade-off"),
    ("CUX2 (chr12 SH2B3/ALDH2 region)", "12_111269073_C_T", "diabetes up, coronary down"),
    ("TCF7L2", "10_112998590_C_T", "the canonical type 2 diabetes allele — concordant, both up"),
    ("CDKN2B (9p21)", "9_22125504_G_C", "the shared cardiometabolic locus — concordant, both up"),
    ("HNF1A", "12_120978847_A_C", "diabetes up and coronary up"),
    ("LPL", "8_19973410_C_T", "both down"),
]

representative_rows = []
for label, variant_id, why in REPRESENTATIVES:
    rows = classified_broad[(classified_broad["variantId"] == variant_id) & ~classified_broad["joint_study"]].copy()
    rows.insert(0, "label", label)
    representative_rows.append(rows)

    t2d_side = rows[rows["class"] == "t2d"]
    coronary_side = rows[rows["class"] == "coronary"]
    print("=" * 110)
    print(
        f"{label}  {variant_id}   effect allele {rows['effectAllele'].iloc[0]} "
        f"(other allele {rows['otherAllele'].iloc[0]})  — {why}"
    )
    print(
        f"  concordance over all {cs.loc[cs['variantId'] == variant_id, 'studyLocusId'].nunique()} "
        f"credible sets of this variant (paper formula): "
        f"{paper_concordance(cs[cs['variantId'] == variant_id]):.3f}"
    )
    print(
        f"  diabetes side: {t2d_side['studyLocusId'].nunique()} credible sets, "
        f"{t2d_side['diseaseId'].nunique()} diseases, sign {sign_of(t2d_side['beta'])}, "
        f"median beta {median_beta(t2d_side['beta']):.3f}"
    )
    print(
        f"  coronary side: {coronary_side['studyLocusId'].nunique()} credible sets, "
        f"{coronary_side['diseaseId'].nunique()} diseases, sign {sign_of(coronary_side['beta'])}, "
        f"median beta {median_beta(coronary_side['beta']):.3f}"
    )
    print()
    print(
        rows.groupby(["class", "diseaseId", "diseaseName"])
        .agg(
            credible_sets=("studyLocusId", "nunique"),
            n_increases=("beta", lambda s: int((s > 0).sum())),
            n_decreases=("beta", lambda s: int((s < 0).sum())),
            min_beta=("beta", "min"),
            max_beta=("beta", "max"),
        )
        .round(3)
        .to_string()
    )
    print()

representatives = pd.concat(representative_rows, ignore_index=True)
representatives = representatives[
    [
        "label",
        "variantId",
        "effectAllele",
        "otherAllele",
        "class",
        "studyId",
        "trait",
        "diseaseId",
        "diseaseName",
        "originalBeta",
        "originalStandardError",
        "beta",
        "se",
        "nCases",
        "nControls",
        "studyLocusId",
    ]
]
representatives.to_csv(INTERMEDIATE + "t2d_cad_representatives-r1.csv", index=False)
print(f"written: {len(representatives)} association rows for {len(REPRESENTATIVES)} representatives")

APOE (rs429358 region)  19_44908684_T_C   effect allele C (other allele T)  — lowers type 2 diabetes risk, raises coronary risk — the referee's pattern
  concordance over all 191 credible sets of this variant (paper formula): 0.659
  diabetes side: 16 credible sets, 4 diseases, sign -, median beta -0.060
  coronary side: 14 credible sets, 6 diseases, sign +, median beta 0.077

                                                    credible_sets  n_increases  n_decreases  min_beta  max_beta
class    diseaseId     diseaseName                                                                             
coronary EFO_0000319   cardiovascular disease                   1            1            0     0.049     0.049
         EFO_0000612   myocardial infarction                    4            4            0     0.060     0.150
         EFO_0001645   coronary artery disease                  1            1            0     0.060     0.060
         EFO_0003777   heart disease                        

# What this shows

**PCSK9 cannot answer the referee's question.** 109 credible sets across 24 lead variants have PCSK9
as their L2G-prioritised gene. **None** of them is mapped to type 2 diabetes or to any glycaemic term
— not under the primary class, not under the broad one that also admits unspecified diabetes mellitus
and the diabetic complications. Its 40 coronary credible sets (48 broad) all point the protective way
for the LDL-lowering alleles. The prediction "the alleles at PCSK9 that increase a diagnosis of
hypercholesterolemia should increase the risk of T2D" is untestable here because the diabetes side of
the locus was never called — missing data, not a concordant result.

**Genes.** 160 L2G-prioritised genes have at least one type 2 diabetes credible set and at least one
coronary credible set (205 under the broad classes) — TCF7L2, CDKN2B, CDKAL1, IGF2BP2, FTO, HHEX,
CCND2 at the top. As on the immunity axis, gene-level overlap is common.

**Allele-exact overlap is much rarer.** Only **27** gene × lead-variant pairs (23 genes) carry both
classes on the same lead variant, 37 pairs (29 genes) under the broad classes. Four trans-disease
studies — "Type 2 diabetes mellitus or coronary artery disease (pleiotropy)", "Cardiometabolic
multimorbidity", "Coronary heart disease x type 2 diabetes interaction" — account for 90 credible
sets that touch both classes at once and are excluded from the direction comparison, since they would
put the identical beta on both sides.

**Verdicts.** Primary classes: 15 concordant, 3 discordant, 9 undetermined (16.7% of determined are
discordant). Broad classes: 25 concordant, 4 discordant, 8 undetermined (13.8%). So on this axis the
dominant pattern really is concordance — cardiometabolic risk moves together at most shared loci —
but the antagonistic pattern the referee expects is present and detected where it exists:

| gene | variant | effect allele | diabetes side | coronary side | verdict |
|---|---|---|---|---|---|
| APOE | `19_44908684_T_C` | C | 16 credible sets, median −0.06 (T2D, diabetes mellitus, diabetic retinopathy, T2D nephropathy) | 14 credible sets, median +0.08 (MI, CAD, angina, coronary atherosclerosis) | **discordant** |
| APOE | `19_44919689_A_G` | G | 1 credible set, −0.06 | 5 credible sets, +0.06 | **discordant** |
| PNPLA3 | `22_43928850_C_T` | T | 3 credible sets, +0.06 | 2 credible sets, −0.03 | **discordant** |
| CUX2 (chr12 SH2B3/ALDH2) | `12_111269073_C_T` | T | 1 credible set, +0.03 | 4 credible sets, −0.13 | **discordant** |
| TCF7L2 | `10_112998590_C_T` | T | 76 credible sets, +0.25 | 8 credible sets, +0.07 | concordant |
| CDKN2B (9p21) | `9_22125504_G_C` | C | 1 credible set, +0.07 | 16 credible sets, +0.16 | concordant |
| HNF1A | `12_120978847_A_C` | C | 2 credible sets, +0.04 | 11 credible sets, +0.05 | concordant |
| LPL | `8_19973410_C_T` | T | 2 credible sets, −0.04 | 1 credible set, −0.07 | concordant |

APOE is the clean example of exactly the mechanism the referee describes — the allele that raises LDL
and coronary risk lowers type 2 diabetes risk — and it is picked up without any special handling: its
paper-formula concordance is **0.659**, one of the lowest of any highly pleiotropic lead variant.

**Caveats.** The primary coronary class inherits `EFO_0010820` (spontaneous coronary artery
dissection) as a descendant of coronary artery disease, which is a different pathology; it contributes
29 credible sets. The diabetes side is overwhelmingly one term (type 2 diabetes, 4,757 credible sets),
so "diabetes" here is a diagnosis label, not a glycaemic measurement — the measurement traits that
would show the LDL–insulin-secretion trade-off most directly are outside this disease-only corpus.